# Build Gold Datasets

Run this notebook after canonical Raw and Silver files are available.

**Input:** canonical files created or checked by `00_data_ingestion.ipynb`.  
**Output:** current product-ready Gold tables.  
**Transformation code:** `src/dataset/`.

This notebook only orchestrates the local pipeline; it does not train models.


## Colab setup

Mount Drive and load the current repository code.


In [ ]:
%pip install -q pyarrow openpyxl

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/kdnehihi/nba-scout-assistant.git"
REPO_DIR = Path("/content/nba-scout-assistant")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


## Resolve data paths

The selected folder must contain `raw/`, `silver/`, and `gold/`.


In [ ]:
from src.dataset.loaders import resolve_data_paths
from src.dataset.pipeline import build_all_gold_datasets

DATA_DIR = Path("/content/drive/MyDrive/nba-scout-assistant/data")
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data folder not found: {DATA_DIR}")

paths = resolve_data_paths(DATA_DIR)
print("Data root:", paths.data_dir)
print("Raw:", paths.raw_dir)
print("Silver:", paths.silver_dir)
print("Gold:", paths.gold_dir)


## Build current Gold tables

`build_all_gold_datasets` performs complete-season filtering, then builds role features, short-term training rows, salary history context, long-term training anchors, long-term inference anchors, and season coverage.


In [ ]:
import pandas as pd

outputs = build_all_gold_datasets(paths)

gold_catalog = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(dataframe),
        "columns": dataframe.shape[1],
        "path": str(paths.gold_dir / f"{name}.parquet"),
    }
    for name, dataframe in outputs.items()
])
display(gold_catalog)


## Next step

Run `02_data_visualization.ipynb` to inspect before/after schemas. Run `03_feature_target_exploration.ipynb` only when feature relationships need to be reviewed again.
